# ⚠️ REQUIRED BEFORE RUNNING — Enable Free GPU

**If you skip this step, the notebook will crash or take 30+ minutes.**

1. Click **Runtime** in the menu above
2. Click **Change runtime type**
3. Set **Hardware accelerator** to **A100 GPU**
4. Click **Save**

Then: **Runtime → Run all** (Ctrl+F9 / Cmd+F9)

---

# TunedAI Labs — Raw Passage Causal Reasoning Test

This notebook tests four AI models on **verbatim passages** from books published before computers existed.
Every passage below was retrieved directly from Project Gutenberg. No AI wrote these sentences.
The correct answers were established by historians, philosophers, and scientists — decades or centuries ago.

| Model | What it is |
|---|---|
| Base Qwen 2.5-7B | An unmodified open-source model |
| GPT-4o | OpenAI's best model (optional — needs API key) |
| Claude 3.5 Sonnet | Anthropic's best model (optional — needs API key) |
| **TunedAI Labs Causal Model** | The same Qwen 2.5-7B, fine-tuned by TunedAI Labs on causal reasoning |

---

### Sources used

| Author | Work | Year | Retrieved from |
|---|---|---|---|
| David Hume | An Enquiry Concerning Human Understanding | 1748 | Project Gutenberg #9662 |
| John Snow | On the Mode of Communication of Cholera (2nd ed.) | 1855 | Project Gutenberg #72894 |
| John Stuart Mill | A System of Logic | 1843 | Project Gutenberg #27942 |
| Florence Nightingale | Notes on Nursing | 1860 | Project Gutenberg #17366 |

These texts were written 85–278 years before the first AI language model. The sentence structures are provably pre-AI.
Our full benchmark (96.96% on 10,112 questions) uses the same reasoning applied across a much wider range of problems.


---
## Cell 1 of 4 — Install required packages

This installs the software needed to run the models. You will see a lot of text scroll by — that is normal. Wait for it to finish before the next cell runs.

In [ ]:
!pip install -q transformers peft accelerate bitsandbytes torch openai anthropic

---
## Cell 2 of 4 — Optional: Add your API keys

If you want to include **GPT-4o** and **Claude 3.5** in the comparison, paste your API keys below between the quotes.

- OpenAI key: get one at **platform.openai.com** → API Keys
- Anthropic key: get one at **console.anthropic.com** → API Keys

**If you leave these blank, those columns will be skipped.** Base Qwen vs TunedAI Labs still runs with no keys needed.

In [ ]:
OPENAI_API_KEY    = ""   # paste your OpenAI key here (optional)
ANTHROPIC_API_KEY = ""   # paste your Anthropic key here (optional)

---
## Cell 3 of 4 — Load the models

This downloads and loads two versions of the same AI model:
- The original unmodified version (Base Qwen 2.5-7B)
- TunedAI Labs fine-tuned version with causal reasoning training

**This takes about 2 minutes.** You will see messages like "Loading tokenizer", "Loading base model", "Loading TunedAI Labs adapter". Wait for the green checkmark ✓ before continuing.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import openai, anthropic

BASE_MODEL   = "Qwen/Qwen2.5-7B-Instruct"
ADAPTER_REPO = "tunedailabs/causal-reasoning-qwen-7b"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

print("Loading base model (~90 seconds on A100)...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="cuda:0",
    trust_remote_code=True,
)

print("Loading TunedAI Labs causal reasoning adapter...")
tuned_model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)
tuned_model.eval()

oai_client = openai.OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None
ant_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY) if ANTHROPIC_API_KEY else None

print("\n✓ All models ready. Scroll down to see the results.")


---
## Cell 4 of 4 — Helper functions

This sets up the comparison engine. It runs automatically — nothing to change here.

In [ ]:
import re
SYSTEM = "You are a careful reasoner. Answer questions about causation, association, intervention, and counterfactuals precisely and correctly."

# Score tracking
_scores = {"base": [], "tuned": [], "questions": 0}

# Keyword groups for each of the 30 questions (3 groups each = 90 total points)
_keywords = [
    [["cause and effect","causal","cause-and-effect"],["beyond","memory","senses","absent"],["letter","reason","evidence","other fact"]],
    [["a priori","reason alone","without experience"],["experience","constantly conjoined","conjunction"],["new object","thought experiment","cannot discover"]],
    [["custom","habit","habitual"],["reason","reasoning","cannot","incapable"],["thousand instances","one instance","repetition","constant conjunction"]],
    [["loose and separate","following another","sequence"],["force","power","tie","mechanism","cannot observe"],["conjoined","connected","distinction"]],
    [["imagination","thought","mind","feeling","mental","habit"],["nothing","physical world","objects","unchanged"],["foretell","expect","inference","predict"]],
    [["habit","custom","carried by habit"],["nothing different","external","physical","same"],["internal","feeling","sentiment","impression"]],
    [["regularity","constant conjunction","always conjoined"],["counterfactual","had not been","never had existed"],["equivalent","not equivalent","differ","overdetermination"]],
    [["voluntary motion","limb","will","volition"],["muscles","nerves","intermediate","mediated"],["mysterious","unknown","hidden","not directly"]],
    [["airborne","miasma","atmosphere","lungs","rules out"],["swallowed","ingested","alimentary canal","stomach"],["mechanistic","site of action","route","eliminative"]],
    [["material","morbid material","passes from","transmitted"],["physical proof","direct evidence","syphilis","smallpox"],["extension","spread","multiplying","indirect"]],
    [["cleanliness","scarcity of water","hygiene","unclean"],["unexplained","till lately","previously"],["mechanism","fecal","hands","food","transfer"]],
    [["elimination","no other","only agent","pattern"],["physical test","chemical test","impurity","clean water"],["epidemiological","pattern","probative","outweighed"]],
    [["same houses","same people","no difference","equivalent"],["without choice","without knowledge","did not choose"],["confound","wealth","socioeconomic","would differ"]],
    [["alone","not prove","does not prove","insufficient"],["confound","confounding","other factors"],["combination","together","ruled out","eliminat"]],
    [["1849","before","same districts","previously"],["water source","Lambeth","changed","moved"],["same population","confound","only variable","natural experiment"]],
    [["intervention","intervening","act on","causal variable"],["correlational","observational","further study"],["test","operationalize","reproducible","causal model"]],
    [["only one circumstance","single common","shared","all instances agree"],["food","example","sick","applied"],["limitation","confounding","multiple","cannot rule out"]],
    [["every circumstance","save one","only one difference","controlled"],["stronger","stronger than","better than","more powerful"],["controls","eliminates","rules out","all other variables"]],
    [["before","after","two instances","fullness of life"],["all circumstances","same except","one thing changed","wound"],["daily life","everyday","constantly","basic structure"]],
    [["experiment","artificial","controlled","manipulate"],["impossible","cannot experiment","observational","naturally occurring"],["exactly similar","must be the same","all circumstances","constraint"]],
    [["alkali","oil","antecedent","cause","contact"],["soap","saponaceous","effect","phenomenon"],["varying","several varieties","nothing else","only common factor"]],
    [["varies together","covariation","correlation","when one changes"],["common cause","third variable","both effects","confound"],["does not prove","direct causation","could be","alternative"]],
    [["intervention","manipulate","increasing","asymmetry","direction"],["do(","do-calculus","intervene","one way","asymmetric"],["cause not effect","heat is the cause","direction of causation"]],
    [["subtract","remainder","residue","remaining","what\u2019s left"],["Neptune","planet","Uranus","perturbation","unknown cause"],["scientific discovery","points to","unobserved","unknown quantity"]],
    [["earlier","antecedent","unnoticed","weeks months years","prior"],["proximate","distal","confusion","not the cause","effect not cause"],["trace back","origin","reparative","symptom","remedy"]],
    [["attribution error","misidentify","not the disease","something quite different"],["fresh air","light","warmth","cleanliness","diet","environment"],["intervene","can fix","clinically","treatment","reduce suffering"]],
    [["hygiene","cleanliness","ventilation","household","cause"],["hospital","does not address","does not target","treats symptoms"],["target the cause","intervene on cause","must address","prevent"]],
    [["effluvia","air","contaminates","impurity","evaporate"],["remove","intervention","policy","follows from"],["mechanism","derived from","causal","not just a rule"]],
    [["wrong model","incorrect","cold air","temperature","confused"],["foul air","putrescing","contaminated","air quality","correct cause"],["consequence","harmful","delay recovery","acting on wrong"]],
    [["confound","case mix","patient mix","different types"],["age","sex","disease","adjust","control for"],["old men","dropsies","young women","consumptions","baseline"]],
]

def _score(answer, q_idx):
    if q_idx >= len(_keywords):
        return 0, 3
    ans = answer.lower()
    hits = sum(1 for grp in _keywords[q_idx] if any(kw in ans for kw in grp))
    return hits, len(_keywords[q_idx])

def ask_local(question, use_adapter=True, max_new_tokens=350):
    if use_adapter:
        tuned_model.enable_adapter_layers()
    else:
        tuned_model.disable_adapter_layers()
    messages = [{"role":"system","content":SYSTEM},{"role":"user","content":question}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = tuned_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

def ask_gpt4(question):
    if not oai_client:
        return "[Skipped — no OpenAI API key provided]"
    r = oai_client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role":"system","content":SYSTEM},{"role":"user","content":question}],
        max_tokens=400
    )
    return r.choices[0].message.content.strip()

def ask_claude(question):
    if not ant_client:
        return "[Skipped — no Anthropic API key provided]"
    r = ant_client.messages.create(
        model="claude-3-5-sonnet-20241022",
        max_tokens=400,
        system=SYSTEM,
        messages=[{"role":"user","content":question}]
    )
    return r.content[0].text.strip()

def compare_all(question, source="", note=""):
    SEP = "=" * 70
    DIV = "-" * 70
    q_idx = _scores["questions"]
    _scores["questions"] += 1
    print(SEP)
    print(f"TEST {q_idx+1}/30")
    if source: print(f"SOURCE : {source}")
    print(SEP)
    print(f"QUESTION:\n{question}\n")

    base_ans  = ask_local(question, use_adapter=False)
    tuned_ans = ask_local(question, use_adapter=True)

    b_hits, b_tot = _score(base_ans,  q_idx)
    t_hits, t_tot = _score(tuned_ans, q_idx)
    _scores["base"].append((b_hits, b_tot))
    _scores["tuned"].append((t_hits, t_tot))

    print(DIV)
    print(f"[ BASE QWEN 2.5-7B (untuned) ]  score: {b_hits}/{b_tot}")
    print(DIV)
    print(base_ans)
    print()

    for label, fn in [("GPT-4o", ask_gpt4), ("CLAUDE 3.5 SONNET", ask_claude)]:
        print(DIV)
        print(f"[ {label} ]")
        print(DIV)
        print(fn(question))
        print()

    print(DIV)
    print(f"[ TUNEDAI CAUSAL MODEL ★ ]  score: {t_hits}/{t_tot}")
    print(DIV)
    print(tuned_ans)
    print()

    # Running tally
    b_total = sum(h for h,_ in _scores["base"])
    t_total = sum(h for h,_ in _scores["tuned"])
    b_max   = sum(m for _,m in _scores["base"])
    print(f"  Running: Base {b_total}/{b_max} — TunedAI {t_total}/{b_max}")

print("\u2713 Ready — running all 30 tests now...")


---
# Results

Each test below shows a **verbatim passage** from a book written before computers existed, followed by a causal reasoning question about that passage.
The correct answers were established by historians, philosophers, and scientists — not by any AI system.

**BASE QWEN** = unmodified open-source model. **TUNEDAI LABS** = the same model after fine-tuning.

---

---
## Test 1 — Hume: §III

**Source:** David Hume, An Enquiry Concerning Human Understanding, §III, 1748  
**Retrieved from:** Project Gutenberg #9662 (verbatim — written before AI existed)


In [ ]:
compare_all(
    question="""
VERBATIM PASSAGE (David Hume, 1748):

"Beyond what we currently see and remember, it is only from the relation of cause and effect, that we can draw any inference. And as this relation can only be discovered by experience, it is experience alone that gives authority to human testimony."

QUESTION: Hume says cause-and-effect is the only relation that lets us reason beyond what we currently see and remember. Give a concrete example from the passage and explain the structure of that reasoning.
""")


---
## Test 2 — Hume: Section IV §23

**Source:** David Hume, An Enquiry Concerning Human Understanding, Section IV §23, 1748  
**Retrieved from:** Project Gutenberg #9662 (verbatim — written before AI existed)


In [ ]:
compare_all(
    question="""
VERBATIM PASSAGE (David Hume, 1748):

"I shall venture to affirm, as a general proposition, which admits of no exception, that the knowledge of this relation is not, in any instance, attained by reasonings a priori; but arises entirely from experience, when we find that any particular objects are constantly conjoined with each other. Let an object be presented to a man of ever so strong natural reason and abilities; if that object be entirely new to him, he will not be able, by the most accurate examination of its sensible qualities, to discover any of its causes or effects."

QUESTION: Hume claims causal knowledge cannot be attained 'a priori.' What does this mean, and what does he say is the actual source of our causal knowledge? What is the thought experiment he uses as evidence?
""")


---
## Test 3 — Hume: Section V

**Source:** David Hume, An Enquiry Concerning Human Understanding, Section V, 1748  
**Retrieved from:** Project Gutenberg #9662 (verbatim — written before AI existed)


In [ ]:
compare_all(
    question="""
VERBATIM PASSAGE (David Hume, 1748):

"after the constant conjunction of two objects—heat and flame, for instance, weight and solidity—we are determined by custom alone to expect the one from the appearance of the other. This hypothesis seems even the only one which explains the difficulty, why we draw, from a thousand instances, an inference which we are not able to draw from one instance, that is, in no respect, different from them."

QUESTION: Hume distinguishes what we conclude from one instance vs. a thousand. What explains the difference, and why does he say 'reason' cannot account for it?
""")


---
## Test 4 — Hume: Section VII Part II §58

**Source:** David Hume, An Enquiry Concerning Human Understanding, Section VII Part II §58, 1748  
**Retrieved from:** Project Gutenberg #9662 (verbatim — written before AI existed)


In [ ]:
compare_all(
    question="""
VERBATIM PASSAGE (David Hume, 1748):

"It appears that, in single instances of the operation of bodies, we never can, by our utmost scrutiny, discover any thing but one event following another, without being able to comprehend any force or power by which the cause operates, or any connexion between it and its supposed effect. All events seem entirely loose and separate. One event follows another; but we never can observe any tie between them. They seem conjoined, but never connected."

QUESTION: What does Hume say we can and cannot observe? What is the significance of the distinction between 'conjoined' and 'connected'?
""")


---
## Test 5 — Hume: Section VII Part II §59 (billiard balls)

**Source:** David Hume, An Enquiry Concerning Human Understanding, Section VII Part II §59 (billiard balls), 1748  
**Retrieved from:** Project Gutenberg #9662 (verbatim — written before AI existed)


In [ ]:
compare_all(
    question="""
VERBATIM PASSAGE (David Hume, 1748):

"The first time a man saw the communication of motion by impulse, as by the shock of two billiard balls, he could not pronounce that the one event was connected: but only that it was conjoined with the other. After he has observed several instances of this nature, he then pronounces them to be connected. What alteration has happened to give rise to this new idea of connexion? Nothing but that he now feels these events to be connected in his imagination."

QUESTION: After observing several billiard ball collisions, a man says they are 'connected.' According to Hume, what has actually changed between the first observation and the several observations — and what has NOT changed?
""")


---
## Test 6 — Hume: Section VII Part II §59

**Source:** David Hume, An Enquiry Concerning Human Understanding, Section VII Part II §59, 1748  
**Retrieved from:** Project Gutenberg #9662 (verbatim — written before AI existed)


In [ ]:
compare_all(
    question="""
VERBATIM PASSAGE (David Hume, 1748):

"It appears, then, that this idea of a necessary connexion among events arises from a number of similar instances which occur of the constant conjunction of these events; nor can that idea ever be suggested by any one of these instances, surveyed in all possible lights and positions. But there is nothing in a number of instances, different from every single instance, which is supposed to be exactly similar; except only, that after a repetition of similar instances, the mind is carried by habit, upon the appearance of one event, to expect its usual attendant."

QUESTION: Hume says 'necessary connexion' arises from repetition — but also that nothing in a thousand instances differs from one. How does he resolve this? What IS different after many repetitions?
""")


---
## Test 7 — Hume: Section VII Part II §60 (counterfactual)

**Source:** David Hume, An Enquiry Concerning Human Understanding, Section VII Part II §60 (counterfactual), 1748  
**Retrieved from:** Project Gutenberg #9662 (verbatim — written before AI existed)


In [ ]:
compare_all(
    question="""
VERBATIM PASSAGE (David Hume, 1748):

"Similar objects are always conjoined with similar. Of this we have experience. Suitably to this experience, therefore, we may define a cause to be an object, followed by another, and where all the objects similar to the first are followed by objects similar to the second. Or in other words where, if the first object had not been, the second never had existed."

QUESTION: Hume offers two definitions of cause. State both. Are they equivalent? Give an example where one gives a clear answer and the other becomes difficult to apply.
""")


---
## Test 8 — Hume: Section VII Part I

**Source:** David Hume, An Enquiry Concerning Human Understanding, Section VII Part I, 1748  
**Retrieved from:** Project Gutenberg #9662 (verbatim — written before AI existed)


In [ ]:
compare_all(
    question="""
VERBATIM PASSAGE (David Hume, 1748):

"experience only teaches us, how one event constantly follows another; without instructing us in the secret connexion, which binds them together, and renders them inseparable. Thirdly, We learn from anatomy, that the immediate object of power in voluntary motion, is not the member itself which is moved, but certain muscles, and nerves, and animal spirits, and, perhaps, something still more minute and more unknown, through which the motion is successively propagated."

QUESTION: Hume uses voluntary motion (moving a limb) to make a point about causal power. What is his argument? What does anatomy reveal that supports his conclusion?
""")


---
## Test 9 — Snow: 2nd ed.

**Source:** John Snow, On the Mode of Communication of Cholera, 2nd ed., 1855  
**Retrieved from:** Project Gutenberg #72894 (verbatim — written before AI existed)


In [ ]:
compare_all(
    question="""
VERBATIM PASSAGE (John Snow, 1855):

"As cholera commences with an affection of the alimentary canal, and as we have seen that the blood is not under the influence of any poison in the early stages of this disease, it follows that the morbid material producing cholera must be introduced into the alimentary canal—must, in fact, be swallowed accidentally, for persons would not take it intentionally."

QUESTION: Snow uses the anatomy of cholera's early stages to rule out a theory. What theory does he rule out, what does he conclude about the route of infection, and what type of causal reasoning is he using?
""")


---
## Test 10 — Snow: 2nd ed.

**Source:** John Snow, On the Mode of Communication of Cholera, 2nd ed., 1855  
**Retrieved from:** Project Gutenberg #72894 (verbatim — written before AI existed)


In [ ]:
compare_all(
    question="""
VERBATIM PASSAGE (John Snow, 1855):

"Diseases which are communicated from person to person are caused by some material which passes from the sick to the healthy, and which has the property of increasing and multiplying in the systems of the persons it attacks. In syphilis, small-pox, and vaccinia, we have physical proof of the increase of the morbid material, and in other communicable diseases the evidence of this increase, derived from the fact of their extension, is equally conclusive."

QUESTION: Snow makes a general causal claim about communicable diseases. What is the claim? What two types of evidence does he offer, and why does he consider both conclusive?
""")


---
## Test 11 — Snow: 2nd ed.

**Source:** John Snow, On the Mode of Communication of Cholera, 2nd ed., 1855  
**Retrieved from:** Project Gutenberg #72894 (verbatim — written before AI existed)


In [ ]:
compare_all(
    question="""
VERBATIM PASSAGE (John Snow, 1855):

"The instances in which minute quantities of the ejections and dejections of cholera patients must be swallowed are sufficiently numerous to account for the spread of the disease; and on examination it is found to spread most where the facilities for this mode of communication are greatest. Nothing has been found to favour the extension of cholera more than want of personal cleanliness, whether arising from habit or scarcity of water, although the circumstance till lately remained unexplained."

QUESTION: Snow identifies a pattern — cholera spreads most where a certain condition is greatest. What is the condition, what was surprising about this pattern before his theory, and how does his causal theory resolve what 'till lately remained unexplained'?
""")


---
## Test 12 — Snow: 2nd ed.

**Source:** John Snow, On the Mode of Communication of Cholera, 2nd ed., 1855  
**Retrieved from:** Project Gutenberg #72894 (verbatim — written before AI existed)


In [ ]:
compare_all(
    question="""
VERBATIM PASSAGE (John Snow, 1855):

"As soon as I became acquainted with the situation and extent of this irruption of cholera, I suspected some contamination of the water of the much-frequented street pump in Broad Street, near the end of Cambridge Street; but on examining the water, on the evening of the 3rd September, I found so little impurity in it of an organic nature, that I hesitated to come to a conclusion. Further inquiry, however, showed me that there was no other circumstance or agent common to the circumscribed locality in which this sudden increase of cholera occurred, and not extending beyond it, except the water of the above mentioned pump."

QUESTION: Snow hesitated to conclude the pump was the source because the water looked clean. What overrode his hesitation, and what does this tell us about direct physical tests vs. epidemiological pattern evidence in causal reasoning?
""")


---
## Test 13 — Snow: 2nd ed.

**Source:** John Snow, On the Mode of Communication of Cholera, 2nd ed., Chapter IX, 1855  
**Retrieved from:** Project Gutenberg #72894 (verbatim — written before AI existed)


In [ ]:
compare_all(
    question="""
VERBATIM PASSAGE (John Snow, 1855):

"As there is no difference whatever, either in the houses or the people receiving the supply of the two Water Companies, or in any of the physical conditions with which they are surrounded, it is obvious that no experiment could have been devised which would more thoroughly test the effect of water supply on the progress of cholera than this, which circumstances placed ready made before the observer. No fewer than three hundred thousand people of both sexes, of every age and occupation, and of every rank and station, from gentlefolks down to the very poor, were divided into two groups without their choice, and, in most cases, without their knowledge."

QUESTION: Snow claims this natural situation eliminates confounding. List every feature that makes the two groups equivalent. What would be missing if people had chosen their own water supply?
""")


---
## Test 14 — Snow: 2nd ed.

**Source:** John Snow, On the Mode of Communication of Cholera, 2nd ed., 1855  
**Retrieved from:** Project Gutenberg #72894 (verbatim — written before AI existed)


In [ ]:
compare_all(
    question="""
VERBATIM PASSAGE (John Snow, 1855):

"consequently, as 286 fatal attacks of cholera took place, in the first four weeks of the epidemic, in houses supplied by the former Company, and only 14 in houses supplied by the latter, the proportion of fatal attacks to each 10,000 houses was as follows. Southwark and Vauxhall 71. Lambeth 5. The cholera was therefore fourteen times as fatal at this period, amongst persons having the impure water of the Southwark and Vauxhall Company, as amongst those having the purer water from Thames Ditton."

QUESTION: Snow reports a 14-fold difference in mortality. Does this quantitative difference alone prove that the water caused the higher mortality? What additional reasoning is required, and where did Snow provide it?
""")


---
## Test 15 — Snow: 2nd ed.

**Source:** John Snow, On the Mode of Communication of Cholera, 2nd ed., 1855  
**Retrieved from:** Project Gutenberg #72894 (verbatim — written before AI existed)


In [ ]:
compare_all(
    question="""
VERBATIM PASSAGE (John Snow, 1855):

"It thus appears that the districts partially supplied with the improved water suffered much less than the others, although, in 1849, when the Lambeth Company obtained their supply opposite Hungerford Market, these same districts suffered quite as much as those supplied entirely by the Southwark and Vauxhall Company."

QUESTION: Snow compares the same districts in 1849 and the current epidemic. What changed between the two periods, and why is this before-and-after comparison powerful evidence for his causal claim?
""")


---
## Test 16 — Snow: 2nd ed.

**Source:** John Snow, On the Mode of Communication of Cholera, 2nd ed., 1855  
**Retrieved from:** Project Gutenberg #72894 (verbatim — written before AI existed)


In [ ]:
compare_all(
    question="""
VERBATIM PASSAGE (John Snow, 1855):

"I had an interview with the Board of Guardians of St. James's parish, on the evening of Thursday, 7th September, and represented the above circumstances to them. In consequence of what I said, the handle of the pump was removed on the following day."

QUESTION: The pump handle removal is often cited as one of the first public health interventions based on causal reasoning. What type of causal act is this — and what would a pure correlational study have recommended instead? Why does the distinction matter?
""")


---
## Test 17 — Mill: Book III Chapter 8

**Source:** John Stuart Mill, A System of Logic, Book III Chapter 8, First Canon, 1843  
**Retrieved from:** Project Gutenberg #27942 (verbatim — written before AI existed)


In [ ]:
compare_all(
    question="""
VERBATIM PASSAGE (John Stuart Mill, 1843):

"If two or more instances of the phenomenon under investigation have only one circumstance in common, the circumstance in which alone all the instances agree, is the cause (or effect) of the given phenomenon."

QUESTION: State Mill's First Canon (Method of Agreement) in plain language. Give an example of how you would apply it to determine the cause of food poisoning at a restaurant, and identify its main limitation.
""")


---
## Test 18 — Mill: Book III Chapter 8

**Source:** John Stuart Mill, A System of Logic, Book III Chapter 8, Second Canon, 1843  
**Retrieved from:** Project Gutenberg #27942 (verbatim — written before AI existed)


In [ ]:
compare_all(
    question="""
VERBATIM PASSAGE (John Stuart Mill, 1843):

"If an instance in which the phenomenon under investigation occurs, and an instance in which it does not occur, have every circumstance in common save one, that one occurring only in the former; the circumstance in which alone the two instances differ, is the effect, or the cause, or an indispensable part of the cause, of the phenomenon."

QUESTION: State Mill's Second Canon (Method of Difference) in plain language. Why is this method considered stronger than the Method of Agreement for establishing causation?
""")


---
## Test 19 — Mill: Book III Chapter 8

**Source:** John Stuart Mill, A System of Logic, Book III Chapter 8, Second Canon example, 1843  
**Retrieved from:** Project Gutenberg #27942 (verbatim — written before AI existed)


In [ ]:
compare_all(
    question="""
VERBATIM PASSAGE (John Stuart Mill, 1843):

"It is scarcely necessary to give examples of a logical process to which we owe almost all the inductive conclusions we draw in daily life. When a man is shot through the heart, it is by this method we know that it was the gunshot which killed him: for he was in the fullness of life immediately before, all circumstances being the same, except the wound."

QUESTION: Identify the two instances Mill is comparing, what they share, and what differs. Then explain why Mill says this method underlies 'almost all the inductive conclusions we draw in daily life.'
""")


---
## Test 20 — Mill: Book III Chapter 8

**Source:** John Stuart Mill, A System of Logic, Book III Chapter 8, 1843  
**Retrieved from:** Project Gutenberg #27942 (verbatim — written before AI existed)


In [ ]:
compare_all(
    question="""
VERBATIM PASSAGE (John Stuart Mill, 1843):

"Of these methods, that of Difference is more particularly a method of artificial experiment; while that of Agreement is more especially the resource employed where experimentation is impossible. It is inherent in the peculiar character of the Method of Difference, that the nature of the combinations which it requires is much more strictly defined than in the Method of Agreement. The two instances which are to be compared with one another must be exactly similar, in all circumstances except the one which we are attempting to investigate."

QUESTION: Mill distinguishes when Method of Agreement vs. Method of Difference applies. In what situations does each apply, and what practical constraint makes the Method of Difference harder to use in the real world?
""")


---
## Test 21 — Mill: Book III Chapter 8

**Source:** John Stuart Mill, A System of Logic, Book III Chapter 8, Method of Agreement example, 1843  
**Retrieved from:** Project Gutenberg #27942 (verbatim — written before AI existed)


In [ ]:
compare_all(
    question="""
VERBATIM PASSAGE (John Stuart Mill, 1843):

"For example, let the antecedent A be the contact of an alkaline substance and an oil. This combination being tried under several varieties of circumstances, resembling each other in nothing else, the results agree in the production of a greasy and detersive or saponaceous substance: it is therefore concluded that the combination of an oil and an alkali causes the production of a soap. It is thus we inquire, by the Method of Agreement, into the effect of a given cause."

QUESTION: Mill's soap example demonstrates the Method of Agreement. What is the antecedent (cause), what is the phenomenon (effect), and how does varying the other circumstances strengthen the causal conclusion?
""")


---
## Test 22 — Mill: Book III Chapter 8

**Source:** John Stuart Mill, A System of Logic, Book III Chapter 8, Sixth Canon, 1843  
**Retrieved from:** Project Gutenberg #27942 (verbatim — written before AI existed)


In [ ]:
compare_all(
    question="""
VERBATIM PASSAGE (John Stuart Mill, 1843):

"Whatever phenomenon varies in any manner whenever another phenomenon varies in some particular manner, is either a cause or an effect of that phenomenon, or is connected with it through some fact of causation."

QUESTION: State Mill's Method of Concomitant Variations in plain language. Why does Mill include the phrase 'or is connected with it through some fact of causation' — what possibility does that cover?
""")


---
## Test 23 — Mill: Book III Chapter 8

**Source:** John Stuart Mill, A System of Logic, Book III Chapter 8, Concomitant Variations example, 1843  
**Retrieved from:** Project Gutenberg #27942 (verbatim — written before AI existed)


In [ ]:
compare_all(
    question="""
VERBATIM PASSAGE (John Stuart Mill, 1843):

"In the case of heat, for example, by increasing the temperature of a body we increase its bulk, but by increasing its bulk we do not increase its temperature; on the contrary (as in the rarefaction of air under the receiver of an air-pump), we generally diminish it: therefore heat is not an effect, but a cause, of increase of bulk."

QUESTION: Mill uses the heat/bulk example to determine the direction of causation. What logical test does he apply, and how does this test relate to the modern concept of intervention?
""")


---
## Test 24 — Mill: Book III Chapter 8

**Source:** John Stuart Mill, A System of Logic, Book III Chapter 8, Method of Residues, 1843  
**Retrieved from:** Project Gutenberg #27942 (verbatim — written before AI existed)


In [ ]:
compare_all(
    question="""
VERBATIM PASSAGE (John Stuart Mill, 1843):

"Subducting from any given phenomenon all the portions which, by virtue of preceding inductions, can be assigned to known causes, the remainder will be the effect of the antecedents which had been overlooked, or of which the effect was as yet an unknown quantity. Suppose, as before, that we have the antecedents A B C, followed by the consequents a b c... and are thence apprised that the effect of A is a, and that the effect of B is b. Subtracting the sum of these effects from the total phenomenon, there remains c, which now, without any fresh experiments, we may know to be the effect of C."

QUESTION: Describe Mill's Method of Residues in plain language. How has this type of reasoning been applied in real-world scientific discovery?
""")


---
## Test 25 — Nightingale: and What It Is Not

**Source:** Florence Nightingale, Notes on Nursing: What It Is, and What It Is Not, 1860  
**Retrieved from:** Project Gutenberg #17366 (verbatim — written before AI existed)


In [ ]:
compare_all(
    question="""
VERBATIM PASSAGE (Florence Nightingale, 1860):

"Shall we begin by taking it as a general principle--that all disease, at some period or other of its course, is more or less a reparative process, not necessarily accompanied with suffering: an effort of nature to remedy a process of poisoning or of decay, which has taken place weeks, months, sometimes years beforehand, unnoticed, the termination of the disease being then, while the antecedent process was going on, determined?"

QUESTION: Nightingale proposes that disease is a 'reparative process' caused by an earlier, unnoticed antecedent. What does this mean for how we identify the cause of a disease? What error in causal reasoning does she implicitly warn against?
""")


---
## Test 26 — Nightingale: and What It Is Not

**Source:** Florence Nightingale, Notes on Nursing: What It Is, and What It Is Not, 1860  
**Retrieved from:** Project Gutenberg #17366 (verbatim — written before AI existed)


In [ ]:
compare_all(
    question="""
VERBATIM PASSAGE (Florence Nightingale, 1860):

"In watching disease, both in private houses and in public hospitals, the thing which strikes the experienced observer most forcibly is this, that the symptoms or the sufferings generally considered to be inevitable and incident to the disease are very often not symptoms of the disease at all, but of something quite different--of the want of fresh air, or of light, or of warmth, or of quiet, or of cleanliness, or of punctuality and care in the administration of diet, of each or of all of these."

QUESTION: Nightingale identifies a common causal attribution error in medical care. What is the error, what does she say is the actual cause of many 'disease symptoms,' and why is this distinction clinically important?
""")


---
## Test 27 — Nightingale: and What It Is Not

**Source:** Florence Nightingale, Notes on Nursing: What It Is, and What It Is Not, 1860  
**Retrieved from:** Project Gutenberg #17366 (verbatim — written before AI existed)


In [ ]:
compare_all(
    question="""
VERBATIM PASSAGE (Florence Nightingale, 1860):

"The causes of the enormous child mortality are perfectly well known; they are chiefly want of cleanliness, want of ventilation, want of white-washing; in one word, defective household hygiene. The remedies are just as well known; and among them is certainly not the establishment of a Child's Hospital."

QUESTION: Nightingale distinguishes between the known cause of child mortality and an incorrect remedy. What is the cause, what is the incorrectly proposed remedy, and why does she reject it? What principle of causal reasoning does her argument illustrate?
""")


---
## Test 28 — Nightingale: and What It Is Not

**Source:** Florence Nightingale, Notes on Nursing: What It Is, and What It Is Not, 1860  
**Retrieved from:** Project Gutenberg #17366 (verbatim — written before AI existed)


In [ ]:
compare_all(
    question="""
VERBATIM PASSAGE (Florence Nightingale, 1860):

"In laying down the principle that the first object of the nurse must be to keep the air breathed by her patient as pure as the air without, it must not be forgotten that everything in the room which can give off effluvia, besides the patient, evaporates itself into his air. And it follows that there ought to be nothing in the room, excepting him, which can give off effluvia or moisture."

QUESTION: Nightingale derives an intervention policy from a causal principle. State the causal principle, show how the policy follows from it, and explain why this is causal reasoning rather than simple rule-following.
""")


---
## Test 29 — Nightingale: and What It Is Not

**Source:** Florence Nightingale, Notes on Nursing: What It Is, and What It Is Not, 1860  
**Retrieved from:** Project Gutenberg #17366 (verbatim — written before AI existed)


In [ ]:
compare_all(
    question="""
VERBATIM PASSAGE (Florence Nightingale, 1860):

"I have known a medical officer keep his ward windows hermetically closed, thus exposing the sick to all the dangers of an infected atmosphere, because he was afraid that, by admitting fresh air, the temperature of the ward would be too much lowered. This is a destructive fallacy. To attempt to keep a ward warm at the expense of making the sick repeatedly breathe their own hot, humid, putrescing atmosphere is a certain way to delay recovery or to destroy the sick."

QUESTION: Nightingale describes a medical officer acting on an incorrect causal model. What is his causal model, what is the correct causal model, and what is the consequence of acting on the wrong one?
""")


---
## Test 30 — Nightingale: and What It Is Not

**Source:** Florence Nightingale, Notes on Nursing: What It Is, and What It Is Not, 1860  
**Retrieved from:** Project Gutenberg #17366 (verbatim — written before AI existed)


In [ ]:
compare_all(
    question="""
VERBATIM PASSAGE (Florence Nightingale, 1860):

"In comparing the deaths of one hospital with those of another, any statistics are justly considered absolutely valueless which do not give the ages, the sexes, and the diseases of all the cases. It does not seem necessary to say that there can be no comparison between old men with dropsies and young women with consumptions. Yet the cleverest men and the cleverest women are often heard making such comparisons, ignoring entirely sex, age, disease, place--in fact, all the conditions essential to the question. It is the merest gossip."

QUESTION: Nightingale calls unadjusted hospital mortality comparisons 'absolutely valueless.' Name the statistical problem she identifies, list the variables she says must be controlled for, and explain with her own example why ignoring them misleads.
""")


---
## Final Results — All 30 Questions

Keyword scoring across all 30 verbatim passage questions. 3 keyword groups per question = 90 total points.

In [ ]:
# Final scoreboard
b_total = sum(h for h,_ in _scores["base"])
t_total = sum(h for h,_ in _scores["tuned"])
maximum = sum(m for _,m in _scores["base"])
b_pct = 100*b_total/maximum if maximum else 0
t_pct = 100*t_total/maximum if maximum else 0

print("=" * 70)
print("FINAL RESULTS — TunedAI Labs Raw Passage Test")
print("30 verbatim passages · Hume/Snow/Mill/Nightingale · 3 keyword groups each")
print("=" * 70)
print(f"  Base Qwen 2.5-7B (untuned) : {b_total:3d}/{maximum} = {b_pct:.1f}%")
print(f"  TunedAI Causal Model ★     : {t_total:3d}/{maximum} = {t_pct:.1f}%")
print(f"  Improvement                : +{t_pct-b_pct:.1f} percentage points")
print("=" * 70)
print()
print("Per-question breakdown:")
print(f"  {'Q':<4} {'Base':>6} {'Tuned':>6}")
print(f"  {'-'*4} {'-'*6} {'-'*6}")
for i,(bh,bm) in enumerate(_scores["base"]):
    th,tm = _scores["tuned"][i]
    flag = " ★" if th > bh else ("  " if th == bh else " ↓")
    print(f"  Q{i+1:<3} {bh}/{bm}    {th}/{tm}{flag}")


---
## Try Your Own Question

Replace the text below with any causal reasoning question. Examples to try:
- "Does wearing a seatbelt cause people to drive more recklessly?"
- "If a rooster crows before sunrise every day, does the rooster cause the sun to rise?"
- "A city adds more police and crime goes up. Does adding police cause more crime?"

In [ ]:
MY_QUESTION = """
Type your question here and run this cell.
"""

compare_all(MY_QUESTION, source="Custom")

---
## Share What You Saw

Open a [GitHub Issue](https://github.com/tunedailabs/tunedailabs/issues/new) and paste your results.

Tell us which questions the TunedAI Labs model got right that the others did not — or anything surprising you found.

We read every submission.

---

**TunedAI Labs** — We fine-tune open-source LLMs for real-world reasoning.

[tunedailabs.com](https://tunedailabs.com)